# 01 — Preprocessing and data quality

**MekongDS: from river forecast to farm action.**

This notebook turns the raw `dataset_export/` files into clean, analysis-ready tables:

- daily stage per station on a complete calendar, with a documented reliability mask
- commune exposure attributes (elevation, station assignment, gauge datum)
- live 15-minute telemetry for the 4 live stations
- the 23 production DANGER alerts joined to commune metadata

Every cleaning decision is stated explicitly and justified. Outputs go to
`data/processed/` as parquet for the downstream notebooks.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

from mekongds import data
from mekongds.plotting import set_style

set_style()
pd.set_option("display.width", 160)

RAW = data.data_dir()
OUT = data.processed_dir()

## 1. Daily stage record (MRC RFMMC, 22 stations, 1992–2026)

Raw file: `historical/mrc_daily_all.csv`, 172,829 rows, one per station-day,
stage in meters above local gauge zero.

In [2]:
stage = data.load_stage()
print(f"rows={len(stage):,}  stations={stage.station.nunique()}  "
      f"range {stage.date.min().date()} → {stage.date.max().date()}")

rows=172,829  stations=22  range 1992-06-01 → 2026-10-25


### 1.1 Integrity checks

Checks: duplicate keys, nulls, impossible values, threshold consistency,
calendar coverage. Results feed the cleaning policy below.

In [3]:
qa = {}
qa["duplicate (station, date) keys"] = stage.duplicated(["station", "date"]).sum()
qa["null stage values"] = stage.stage.isna().sum()
qa["negative stage values"] = (stage.stage < 0).sum()
th = stage.groupby("station")[["floodStage", "alarmStage"]].nunique()
qa["stations with inconsistent thresholds"] = int((th > 1).any(axis=1).sum())
qa = {k: int(v) for k, v in qa.items()}
qa

{'duplicate (station, date) keys': 0,
 'null stage values': 0,
 'negative stage values': 334,
 'stations with inconsistent thresholds': 0}

In [4]:
neg = stage[stage.stage < 0].groupby("station").size().sort_values(ascending=False)
neg

station
TCH    166
CDO    154
SAV     14
dtype: int64

**Findings.**

- No duplicates, no nulls, thresholds consistent for all 22 stations. The export is
  already well-formed at the row level.
- 334 negative stages, all at the Vietnamese delta stations (TCH, CDO) and one
  Lao station (SAV). These are deltas relative to a local gauge zero; tides push
  the delta stations below gauge zero in the dry season. **Decision: keep them.**
  They are real, and none of our target stations are affected.

### 1.2 Calendar coverage and the reliability mask

The record is not uniformly complete: 1992–2000 contain only reference wet years
(~214 days), 2001–2006 are partial, the 2014 wet season is missing at source
(the RFMMC 2014 wet column was zeroed), and the last year is in progress.
Annual-maxima and trend analyses must know this.

In [5]:
w = data.stage_wide()
cov = w.notna().groupby(w.index.year).mean()
cov.iloc[:, :11].round(2).style.background_gradient(cmap="Blues", vmin=0, vmax=1)

station,CDO,CKH,CSA,KHO,KOH,KOM,KRA,LUA,MUK,NAK,NEA
date,,,,,,,,,,,
1992,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000
1993,0.000000,0.410000,0.410000,0.410000,0.410000,0.410000,0.410000,0.410000,0.410000,0.410000,0.410000
1994,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
1995,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
1996,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
1997,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
1998,0.420000,0.420000,0.420000,0.420000,0.420000,0.420000,0.420000,0.420000,0.420000,0.420000,0.420000
1999,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
2000,0.420000,0.420000,0.420000,0.420000,0.420000,0.420000,0.420000,0.420000,0.420000,0.420000,0.420000


In [6]:
months_2014 = sorted(stage[stage.date.dt.year == 2014].date.dt.month.unique())
f"months present in 2014: {months_2014}  (Jun–Oct wet season missing at source)"

'months present in 2014: [np.int32(1), np.int32(2), np.int32(3), np.int32(4), np.int32(5), np.int32(11), np.int32(12)]  (Jun–Oct wet season missing at source)'

**Coverage policy.**

- **Reliable era = 2007-01-01 → 2025-12-31**, minus the 2014 wet season
  (2014-06-01 → 2014-10-31). In this window every station has near-complete
  daily coverage (see matrix above; only TCH/CDO have minor gaps).
- 1992–2006 rows are kept in the tidy table but flagged `reliable=False`:
  annual statistics there would be biased toward the wet season because the
  dry-season rows are missing in the reference years.
- 2026 is in progress and is treated as a test/case-study year only.

In [7]:
stage_tidy = stage.copy()
stage_tidy["reliable"] = (
    (stage_tidy.date >= "2007-01-01")
    & (stage_tidy.date <= "2025-12-31")
    & ~((stage_tidy.date >= "2014-06-01") & (stage_tidy.date <= "2014-10-31"))
)
stage_tidy["reliable"].value_counts()

reliable
True     144077
False     28752
Name: count, dtype: int64

### 1.3 Gap structure within the reliable era

How many missing days remain inside the reliable era, and are they isolated
(interpolation is defensible) or block gaps (interpolation would fabricate a flood wave)?

In [8]:
rel = w.loc["2007":"2025"]
gaps = rel.isna().sum().sort_values(ascending=False)
gaps

station
CDO    823
TCH    736
KHO    728
CKH    593
PAK    497
SAV    445
VIE    393
THA    350
PKS    338
PPP    334
LUA    323
STR    315
PPB    286
KRA    284
KOM    284
NEA    276
KOH    274
PRE    273
CSA    270
NON    262
MUK    260
NAK    259
dtype: int64

In [9]:
# longest run of consecutive missing days per station inside the reliable era
def longest_gap(s: pd.Series) -> int:
    isnan = s.isna().to_numpy()
    best = cur = 0
    for v in isnan:
        cur = cur + 1 if v else 0
        best = max(best, cur)
    return best

longest = {c: longest_gap(rel[c]) for c in rel.columns}
pd.Series(longest).sort_values(ascending=False).head(8)

KHO    456
CKH    334
CDO    276
TCH    276
THA    245
SAV    245
PKS    245
PAK    245
dtype: int64

Missing days inside the reliable era are a few hundred per station at most and
runs are short (a handful of days), except TCH/CDO where they are larger but
those are delta reference stations, not forecast targets.

**Interpolation policy.** We do **not** fill gaps in the stage matrix. Model
training uses feature rows only where the needed lags exist (LightGBM also
tolerates missing values natively), and annual-maxima statistics use a
coverage filter instead (notebook 03). Interpolating stage across even 3–4
days can erase the peak of a flood wave, which is exactly the signal we care
about, so we prefer fewer, truthful rows to a smooth, partly invented series.

In [10]:
w.to_parquet(OUT / "stage_wide.parquet")
stage_tidy.to_parquet(OUT / "stage_tidy.parquet", index=False)

## 2. Station metadata and hydrologic order

`stations_historical.csv` lists stations upstream → downstream. We attach the
live-network telemetry ids and the flood/alarm thresholds, and define the
station set used by the forecast model (Cambodian stations, the ones whose
communes are in the gazetteer).

In [11]:
meta = data.load_station_meta()
meta

,station,station_name,telemetry_id,river,country,lat,lng,floodStage,alarmStage,order
0,CSA,Chiang Saen,10501.0,Mekong,Thailand,20.274120,100.088550,12.8,11.5,0
1,CKH,Chiang Khan,11903.0,Mekong,Thailand,17.900260,101.669890,16.0,14.5,1
2,LUA,Luang Prabang,11201.0,Mekong,Lao PDR,19.892800,102.134180,18.0,17.5,2
3,VIE,Vientiane KM4,11901.0,Mekong,Lao PDR,17.930980,102.615560,12.5,11.5,3
4,NON,Nong Khai 2,12002.0,Mekong,Thailand,17.877750,102.716670,12.2,11.4,4
5,PAK,Paksane,12703.0,Mekong,Lao PDR,18.373404,103.663210,14.5,13.5,5
6,NAK,Nakhon Phanom,13101.0,Mekong,Thailand,17.623796,104.517420,12.0,11.5,6
7,THA,Thakhek,13102.0,Mekong,Lao PDR,17.394533,104.801400,14.0,13.0,7
8,SAV,Savannakhet,13401.0,Mekong,Lao PDR,16.563604,105.742970,13.0,12.0,8
9,MUK,Mukdahan,13402.0,Mekong,Thailand,16.582800,104.733180,12.5,12.0,9


In [12]:
# The live network maps to these historical codes
LIVE = {"KOM": "019802", "PRE": "020102", "CHT*": "033401", "KOH": "033402"}
# *Chaktomuk (live 033401) has no historical twin; Phnom Penh Port (PPP) is the
# nearest historical gauge at the same confluence and stands in for it.
live_st = data.load_live_station_thresholds()
live_st[["station", "name_en", "river", "alarm_stage_m", "flood_stage_m", "zero_gauge_elevation_m"]]

,station,name_en,river,alarm_stage_m,flood_stage_m,zero_gauge_elevation_m
0,KOM,Kompong Cham,Mekong,15.2,16.2,NaN
1,PRE,Prek Kdam,Tonlesap,9.5,10.0,0.08
2,CHT,Chaktomuk,Bassac,10.5,12.0,-1.02
3,KOH,Koh Khel,Bassac,7.9,8.4,-1.00


**Gauge datums.** `zero_gauge_elevation_m` is published for PRE (+0.08 m),
Chaktomuk (−1.02 m) and Koh Khel (−1.00 m), but missing for Kompong Cham.
This matters for the decision layer: converting a gauge stage to an absolute
water elevation (and hence to "does this commune flood") is only possible
where the datum is known. The commune mapping in notebook 05 therefore covers
PRE, CHT and KOH (1,116 communes) and excludes KOM.

In [13]:
meta.to_parquet(OUT / "station_meta.parquet", index=False)

## 3. Commune exposure attributes

1,661 communes with coordinates, DEM elevation (Copernicus GLO-90 at the
representative point), nearest MRC station and distance. Cleaning: enforce
string PCODEs, flag the 117 communes without boundaries/station assignment,
and validate the join to station metadata.

In [14]:
cm = data.load_communes()
print(f"rows={len(cm)}  provinces={cm.province_code.nunique()}")
cm.head(3)

rows=1661  provinces=25


,code,name_en,name_kh,district_code,district_en,district_kh,province_code,province_en,province_kh,lat,lng,has_boundary,ground_elevation_m,nearest_station_id,distance_to_station_km
0,010201,Banteay Neang,បន្ទាយនាង,0102,Mangkul Bourei,មង្គលបូរី,01,Banteay Meanchey,បន្ទាយមានជ័យ,13.483002,103.035034,True,11.0,020102,267.41
1,010202,Bat Trang,បត់ត្រង់,0102,Mangkul Bourei,មង្គលបូរី,01,Banteay Meanchey,បន្ទាយមានជ័យ,13.523315,102.955343,True,13.0,020102,276.74
2,010203,Chamnaom,ចំណោម,0102,Mangkul Bourei,មង្គលបូរី,01,Banteay Meanchey,បន្ទាយមានជ័យ,13.452301,102.874791,True,15.0,020102,277.92


In [15]:
qa_cm = {
    "null nearest_station_id": int(cm.nearest_station_id.isna().sum()),
    "null ground_elevation_m": int(cm.ground_elevation_m.isna().sum()),
    "zero ground_elevation_m": int((cm.ground_elevation_m == 0).sum()),
    "duplicate codes": int(cm.code.duplicated().sum()),
}
qa_cm

{'null nearest_station_id': 117,
 'null ground_elevation_m': 117,
 'zero ground_elevation_m': 1,
 'duplicate codes': 0}

In [16]:
# Zero-elevation communes: legitimate (Tonle Sap floodplain islands/marshes) or bad DEM samples?
zero = cm[cm.ground_elevation_m == 0]
print(zero[["name_en", "district_en", "province_en", "nearest_station_id"]]
      .groupby("province_en").size().sort_values(ascending=False).to_string())

province_en
Koh Kong    1


Zero elevations cluster around the Tonle Sap lake and the Bassac floodplain
(Kampong Thom, Pursat, Battambang, Kampong Chhnang). These are real wetland
communes sitting at or near sea level, not DEM artifacts. **Decision: keep.**
They are, in fact, the most flood-exposed communes in the country.

In [17]:
cm["has_station"] = cm.nearest_station_id.notna()
cm_by_station = cm.groupby("nearest_station_id").size().sort_values(ascending=False)
cm_by_station

nearest_station_id
020102    541
019802    428
033402    399
033401    176
dtype: int64

In [18]:
# join station names + datums for the exposure table
st = live_st[["id", "station", "name_en", "alarm_stage_m", "flood_stage_m", "zero_gauge_elevation_m"]]
cm_exp = cm.merge(st, left_on="nearest_station_id", right_on="id", how="left")
cm_exp.to_parquet(OUT / "communes_clean.parquet", index=False)
cm_exp[cm_exp.station.notna()].groupby("station").agg(
    communes=("code", "count"),
    elev_p10=("ground_elevation_m", lambda s: s.quantile(0.10)),
    elev_median=("ground_elevation_m", "median"),
)

,communes,elev_p10,elev_median
station,,,
CHT,176,7.5,12.5
KOH,399,3.0,7.0
KOM,428,11.0,44.5
PRE,541,8.0,17.0


## 4. Live telemetry (15-minute, 4 stations)

`hydro_observations.csv`: 2,628 rows, 2026-10-17 → 2026-10-25. Columns
`rainfall_mm` and `temperature_c` are entirely null in this export; `source`
is all `live` (no stale fallbacks). Rate of rise is derivable from stage, so
we recompute it as a consistency check.

In [19]:
hy = data.load_live_hydro()
print(f"rows={len(hy)}  stations={sorted(hy.station.unique())}  "
      f"{hy.recorded_at.min()} → {hy.recorded_at.max()}")
hy[["water_level_m", "rate_of_rise_m_per_hr"]].describe().round(3)

rows=2628  stations=['CHT', 'KOH', 'KOM', 'PRE']  2026-10-17 18:00:00+00:00 → 2026-10-25 13:45:00+00:00


,water_level_m,rate_of_rise_m_per_hr
count,2628.000,2628.000
mean,5.708,-0.005
std,0.506,0.026
min,2.104,-1.295
25%,5.283,-0.008
50%,5.783,-0.005
75%,6.012,-0.001
max,6.925,0.024


In [20]:
# consistency: reported rate_of_rise vs recomputed from consecutive observations
hy2 = hy.sort_values(["station", "recorded_at"]).copy()
hy2["dt_hr"] = hy2.groupby("station").recorded_at.diff().dt.total_seconds() / 3600
hy2["stage_diff"] = hy2.groupby("station").water_level_m.diff()
hy2["ror_recomputed"] = hy2.stage_diff / hy2.dt_hr
chk = hy2.dropna(subset=["ror_recomputed", "rate_of_rise_m_per_hr"])
err = (chk.ror_recomputed - chk.rate_of_rise_m_per_hr).abs()
f"median abs diff reported vs recomputed rate-of-rise: {err.median():.5f} m/h over {len(chk)} rows"

'median abs diff reported vs recomputed rate-of-rise: 0.00400 m/h over 2624 rows'

In [21]:
# 15-min wide stage matrix
live_wide = hy.pivot_table(index="recorded_at", columns="station", values="water_level_m")
live_wide = live_wide.reindex(
    pd.date_range(live_wide.index.min(), live_wide.index.max(), freq="15min")
)
live_wide.index.name = "recorded_at"
live_wide.isna().sum()

station
CHT    118
KOH     86
KOM     96
PRE     80
dtype: int64

In [22]:
# map live codes to their historical counterparts for model compatibility
CODEMAP = {"KOM": "KOM", "PRE": "PRE", "KOH": "KOH", "CHT": "PPP"}
live_hist = live_wide.rename(columns=CODEMAP)
live_hist = live_hist.resample("D").mean()
live_hist.round(3).tail(6)

station,PPP,KOH,KOM,PRE
recorded_at,,,,
2026-10-20 00:00:00+00:00,5.566,5.163,6.388,5.955
2026-10-21 00:00:00+00:00,5.430,5.057,6.193,5.894
2026-10-22 00:00:00+00:00,5.324,4.961,6.055,5.826
2026-10-23 00:00:00+00:00,5.295,4.947,5.890,5.806
2026-10-24 00:00:00+00:00,5.238,4.913,6.000,5.733
2026-10-25 00:00:00+00:00,5.230,4.902,5.966,5.703


In [23]:
live_wide.to_parquet(OUT / "live_hydro_15min.parquet")
live_hist.to_parquet(OUT / "live_hydro_daily.parquet")

**Note for downstream.** The live window (Oct 17–25, 2026) is a slow recession:
all four stations are falling gently. Production dispatched 23 DANGER alerts
on Oct 21–22 during this recession, which notebook 05 examines critically.

## 5. Production alerts (the system we improve on)

23 DANGER-tier "Emergency harvest acceleration" bulletins dispatched
Oct 21–22, 2026, each tied to a commune and a station. Join to commune
metadata for the comparison in notebook 05.

In [24]:
al = data.load_alerts()
al["station"] = al.station_id.map({"033402": "KOH", "020102": "PRE", "019802": "KOM", "033401": "CHT"})
al_enriched = al.merge(
    cm[["code", "name_en", "district_en", "province_en", "ground_elevation_m",
        "nearest_station_id", "distance_to_station_km"]],
    left_on="commune_id", right_on="code", how="left",
)
al_enriched[["dispatched_at", "station", "name_en", "province_en",
             "ground_elevation_m", "distance_to_station_km"]].head(8)

,dispatched_at,station,name_en,province_en,ground_elevation_m,distance_to_station_km
0,2026-10-21 17:45:19.716814+00:00,PRE,Batheay,Kampong Cham,6.0,20.08
1,2026-10-21 17:45:33.791104+00:00,PRE,Kouk Banteay,Kampong Chhnang,6.0,41.80
2,2026-10-21 17:45:55.795957+00:00,PRE,Preaek Anhchanh,Kandal,6.0,17.96
3,2026-10-21 17:46:15.675469+00:00,PRE,Kampong Luong,Kandal,6.0,4.16
4,2026-10-21 17:46:30.693317+00:00,PRE,Kampong Os,Kandal,6.0,8.24
5,2026-10-21 17:46:47.596606+00:00,PRE,Preaek Ta Teaen,Kandal,6.0,8.40
6,2026-10-22 01:00:17.586403+00:00,KOH,Banteay Daek,Kandal,4.0,20.99
7,2026-10-22 01:00:34.602590+00:00,KOH,Kokir Thum,Kandal,4.0,22.32


In [25]:
print(al_enriched.groupby(["station", "province_en"]).size().to_string())

station  province_en    
KOH      Kandal             12
         Prey Veng           4
         Takeo               1
PRE      Kampong Cham        1
         Kampong Chhnang     1
         Kandal              4


In [26]:
al_enriched.to_parquet(OUT / "alerts_enriched.parquet", index=False)

## 6. Weather forecast snapshots — scope decision

`weather_forecasts.csv` holds province-point precipitation/soil-moisture
forecast snapshots, but only for 2026-10-17 → 2026-10-25 (the deployment
window). There is no historical weather covariate anywhere in the export.

In [27]:
wf = data.load_weather_forecasts()
print(f"rows={len(wf)}  provinces={wf.province_code.nunique()}  "
      f"range {wf.forecast_time.min():%Y-%m-%d} → {wf.forecast_time.max():%Y-%m-%d}")

rows=7065  provinces=25  range 2026-10-17 → 2026-10-25


**Scope decision.** The forecast model (notebook 04) is therefore built on
river-stage dynamics alone: upstream lags, seasonality and memory. This is the
classic river-routing setting and is honest about what the data supports.
Rainfall-driven runoff is the main unmodeled driver, and the quantile
forecasts absorb some of that uncertainty. We state this as a limitation
rather than pretending rainfall features exist.

In [28]:
wf.to_parquet(OUT / "weather_forecasts.parquet", index=False)

## 7. Summary of outputs

| File | Content |
| --- | --- |
| `stage_wide.parquet` | daily stage, date × 22 stations, complete calendar, NaN gaps preserved |
| `stage_tidy.parquet` | tidy stage with `reliable` flag (2007–2025 minus 2014 wet) |
| `station_meta.parquet` | 22 stations, order, thresholds, telemetry ids |
| `communes_clean.parquet` | 1,661 communes + station join, datum, thresholds |
| `live_hydro_15min.parquet` | 15-min stage, 4 live stations, 2026-10-17→25 |
| `live_hydro_daily.parquet` | daily means, live codes mapped to historical codes |
| `alerts_enriched.parquet` | 23 DANGER alerts + commune metadata |
| `weather_forecasts.parquet` | province forecast snapshots (deployment window only) |

**Cleaning decisions recap.**

1. Negative delta-station stages kept (real tidal signal below gauge zero).
2. Reliable era = 2007–2025 minus the 2014 wet season; earlier years flagged, not deleted.
3. No gap interpolation anywhere; analyses filter by coverage instead.
4. Zero-elevation communes kept (real Tonle Sap floodplain communes).
5. Chaktomuk (live) ↔ Phnom Penh Port (historical) stand-in documented.
6. No rainfall features anywhere: none exist in the export; models are stage-dynamics only.